# ChatGeo-Magi

This notebook walks through how ChatGeo-Magi works, step by step. ChatGeo-Magi is a conversational AI agent that answers questions about Earth's magnetic field, runs NOAA calculator APIs, and generates plots for users.
By the end of this notebook you will have built a working copy of the agent and can chat with it directly.

> This notebook accompanies the paper *"ChatGeo-Magi: A Retrieval-Augmented and Tool-Using Language Model for Geomagnetic Data Access and User Support"*. The full production code lives in the `agentic/` folder of this repository.

## 1. What Is an Agent?

Whatever a typical chatbot "knows" comes directly from its training data, which may be outdated, vague, or simply wrong for specialized questions like *"What's the magnetic declination in Boulder right now?"*. An agent is different. Instead of answering immediately, the model runs in a continous loop.

1. Reason: Read the question and decide what to do next
2. Act: Call a tool (search documents, hit an API, draw a plot)
3. Observe: Read the tool's result

This pattern is repeated until the model deems that it can create a proper final answer. This loop is called ReAct (Reason + Act). The important part is that tool results are fed back into the model's context, so the final answer is grounded in tool results, not just memory.

ChatGeo-Magi gives its agent two types of tools:

1. RAG Retriever: Searches a database of trusted geomagnetism documents (WMM technical reports, textbooks, tutorials)
2. NOAA API Tools: Calls the official NOAA magnetic field calculators and plotting functions

The model never runs code or HTTP requests itself. It emits a structured request (e.g., call `noaa_mag_api` with arguments XYZ) which the backend validates and executes, and the result comes back as text. That separation is what keeps the system safe and predictable. The worse the model can do is simply break the API call, and that error would be returned to it to allow for the model to correct itself.

## 2. Setup

Before running this notebook you need:

1. [Ollama](https://ollama.com): Installed and running, as this serves the language model locally. We'll use this command to fetch the model: `ollama pull gpt-oss:20b`
2. Documents for the knowledge base: Put PDFs or text files about geomagnetism into `agentic/data/`. The paper's corpus is listed in its Table 1 (WMM technical report, geomagnetism tutorials, textbooks, etc.). Any relevant PDFs work for learning purposes.
3. API keys (both free) for the live-data tools:
   - A geocoding key from [geocode.maps.co](https://geocode.maps.co): Turns city names into latitude/longitude
   - A NOAA calculator key from the [NOAA Geomagnetic Calculator API page](https://www.ngdc.noaa.gov/geomag/CalcAPI.shtml)

The cell below installs the Python packages from the repository's requirements file. It only needs to run once.

In [ ]:
%pip install -r agentic/requirements.txt

In the production code settings live in `agentic/config.py`. Here, they are plain variables so everything is easily editable from one main area.

The paper uses `NovaSearch/stella_en_1.5B_v5`, which scores well on retrieval benchmarks but is a 1.5 billion parameter download. If you want to learn on a smaller machine, swap in the lightweight alternative there. You can search for other embedding models on [HuggingFace](https://huggingface.co/).

In [ ]:
import os

# Lighter alternatives commented on the sides
# Keep in mind weaker models will produce worse results

# Language model served by Ollama
MODEL_NAME = "gpt-oss:20b" # llama3.1:8b

# Embedding model
EMBEDDING_MODEL = "NovaSearch/stella_en_1.5B_v5" # sentence-transformers/all-MiniLM-L6-v2

# Paths
DATA_DIR = "agentic/data" # put your PDFs / .txt files here
CHROMA_DB_PATH = "./chroma_db" # where the vector database is stored

# API Keys
GEOCODE_API_KEY = os.getenv("GEOCODE_API_KEY", "REPLACEME")
CALC_API_KEY = os.getenv("NOAA_CALC_API_KEY", "REPLACEME")

## 3. Retrieval-Augmented Generation

Language models are trained intially once on a vast swath of text. This is great for general knowledge, but breaks down for specific knowledge about certain texts. RAG fixes this by letting the model see relevant documents before answering.

The pipeline has two phases:

Indexing (done once, before any questions):
1. Load the trusted documents
2. Split them into overlapping chunks (we use 1,000 characters with 200 of overlap, so ideas that span a chunk boundary aren't lost)
3. Convert each chunk into an embedding (a vector of numbers where similar meanings land close together)
4. Store the vectors in a database ([ChromaDB](https://www.trychroma.com/))

Retrieval (at question time):
1. Embed the user's question into the same vector space
2. Find the stored chunks closest to it
3. Hand those chunks to the model as context (inject it right before the user prompt)

First, load the embedding model. The first run downloads it, which can take a few minutes.

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model=EMBEDDING_MODEL)
print("Embedding model loaded.")

Now build the vector store. If a database already exists on disk from a previous run, we reuse it. The process takes a while, so its better to use the cached version if we can.


In [ ]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, TextLoader
from langchain_community.document_loaders.merge import MergedDataLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

vector_store = Chroma(
    collection_name="cgmdocs",
    embedding_function=embeddings,
    persist_directory=CHROMA_DB_PATH,
)

# Only index documents if the store is empty (first run)
if vector_store._collection.count() == 0:
    loaders = [
        DirectoryLoader(DATA_DIR, glob="**/*.txt", loader_cls=TextLoader, show_progress=True),
        DirectoryLoader(DATA_DIR, glob="**/*.pdf", loader_cls=PyPDFLoader, show_progress=True),
    ]
    docs = MergedDataLoader(loaders=loaders).load()

    if not docs:
        print(f"WARNING: no documents found in {DATA_DIR}/.")
        print("The agent will still work for API calculations and plots,")
        print("but it will have no reference documents to retrieve from.")
    else:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            add_start_index=True,
        )
        splits = splitter.split_documents(docs)

        for doc in tqdm(splits, desc="Indexing document chunks"):
            vector_store.add_documents(documents=[doc])
        print(f"Indexed {len(splits)} chunks.")
else:
    print(f"Reusing existing vector store ({vector_store._collection.count()} chunks).")

Here's a key design decision. We could force a retrieval on every question and stuff the results into the prompt. Instead, we wrap the retriever as a tool and let the agent decide when to search. A question like *"plot the field strength in London"* only needs API data, thus the agent can skip retrieval entirely and go straight to the calculator.

The description string below matters more than it looks, as it's given to the model to decide what tool it wants to choose. Creating good prompts and descriptions are essential for allowing an agent to thrive.

In [ ]:
from langchain_core.tools import create_retriever_tool

retriever_tool = create_retriever_tool(
    vector_store.as_retriever(),
    "retrieve_geomag_docs",
    "Search and return information about geomagnetism, from authoritative documents "
    "like the World Magnetic Model technical report.",
)

# This is exactly what the agent sees when it calls the tool
print(retriever_tool.invoke({"query": "What is magnetic declination?"})[:1000])

## 4. NOAA API Tools

Retrieval covers knowledge questions. For data questions, the agent needs to run the NOAA calculators.

A tool in LangChain is just a Python function with the `@tool` decorator. The decorator reads the function's type hints and docstring and turns them into a schema the model can see. When the model wants data, it emits a small JSON message like:

```json
{"name": "noaa_mag_api", "args": {"data_name": "declination", "location": "Boulder, CO"}}
```

The backend validates that message, runs the real function, and sends the return value back into the conversation. We start with two plain helper functions. One to turn a place name into coordinates, and one to parse dates flexibly.


In [ ]:
from typing import Dict, List, Optional, Tuple, Union
from datetime import date
from dateutil.parser import parse
import requests

BASE_URLS = {
    "calculator": "https://www.ngdc.noaa.gov/geomag-web/calculators/calculateIgrfwmm",
    "geocode": "https://geocode.maps.co/search",
}

def geocode(location: str) -> Tuple[bool, Union[Tuple[float, float], str]]:
    """Turn a location string like 'Boulder, CO' into (latitude, longitude)."""
    params = {"q": location, "api_key": GEOCODE_API_KEY}
    data = requests.get(BASE_URLS["geocode"], params=params).json()
    if not data:
        return False, "Location not found"
    return True, (float(data[0]["lat"]), float(data[0]["lon"]))

def parse_date(time_str: Optional[str] = None) -> Tuple[int, int, int]:
    """Convert a date string to (year, month, day). Empty or 'now' means today."""
    if not time_str or time_str.lower() in ["now", "no", ""]:
        today = date.today()
        return today.year, today.month, today.day
    parsed = parse(time_str, fuzzy=True)
    return parsed.year, parsed.month, parsed.day

def get_model_type(year: int) -> str:
    """Historical dates use IGRF; current/future dates use the WMM."""
    if year < 1590:
        raise ValueError("Date must be after 1590 for IGRF calculations")
    return "IGRF" if year < 2025 else "WMM"

Next, the function that actually queries NOAA's calculator, and the first real tool that wraps it.

Notice two things about `noaa_mag_api`:

1. The docstring is written directly for the model. It lists the exact valid component names, because the model has to enter one of them correctly.
2. Errors are returned as strings, not raised as exceptions. If the model picks an invalid component, it gets it back and can correct itself on the next loop iteration. This self-correction is one of the biggest practical advantages of the agentic design.

In [ ]:
from langchain_core.tools import tool

VALID_COMPONENTS = [
    "declination", "inclination", "totalintensity", "horintensity",
    "xcomponent", "ycomponent", "zcomponent",
    "declination_sv", "inclination_sv", "totalintensity_sv", "horintensity_sv",
    "xcomponent_sv", "ycomponent_sv", "zcomponent_sv",
]

def fetch_field_data(location, start_time=None, end_time=None):
    """Call the NOAA calculator API and return its JSON response."""
    success, coords = geocode(location)
    if not success:
        return False, coords

    lat, lon = coords
    start_year, start_month, start_day = parse_date(start_time)

    try:
        model_type = get_model_type(start_year)
    except ValueError as e:
        return False, str(e)

    params = {
        "lat1": lat, "lon1": lon,
        "model": model_type,
        "startYear": start_year, "startMonth": start_month, "startDay": start_day,
        "key": CALC_API_KEY,
        "resultFormat": "json",
    }
    if end_time:
        end_year, end_month, end_day = parse_date(end_time)
        params.update({"endYear": end_year, "endMonth": end_month, "endDay": end_day})

    response = requests.get(BASE_URLS["calculator"], params=params)
    return True, response.json()


@tool
def noaa_mag_api(data_name: str, location: str,
                 start_time: Optional[str] = None, end_time: Optional[str] = None) -> str:
    """
    Fetch magnetic field data from NOAA Magnetic Calculator APIs, from start date
    (default to now) to end date (default to now).
    If you want to call multiple times for different data, do these in separate
    tool steps, don't do them together.
    Please select data name from the following list:

    declination, inclination, totalintensity, horintensity
    xcomponent, ycomponent, zcomponent

    All of these can be appended with _sv to get secular variation data,
    e.g. declination_sv.
    """
    if data_name not in VALID_COMPONENTS:
        return f"Invalid data name: {data_name}. Please choose from: {', '.join(VALID_COMPONENTS)}"

    success, result = fetch_field_data(location, start_time, end_time)
    if not success:
        return f"Could not find {data_name}: {result}"

    results = result.get("result", [])
    if not results:
        return f"No {data_name} data found."

    units = result.get("units", {}).get(data_name, "")

    if len(results) == 1:
        return f"{results[0][data_name]:.2f} {units}"

    lines = []
    for entry in results:
        year = entry.get("year") or entry.get("date") or "unknown"
        lines.append(f"{year}: {entry[data_name]:.2f} {units}")
    return f"{data_name} over time:\n" + "\n".join(lines)

### The plotting tool

The last tool in this example draws a chart. The model can't see or produce images — so the tool saves the figure to disk and returns the file path as text. The interface layer (this notebook, or the Streamlit website in `agentic/cgmwebsite.py`) watches for paths ending in `.png` and renders them for the user.

This tool is deliberately generic, as it it takes raw `x` and `y` lists rather than a location and date range. That means the agent has to chain tools together to plot information. First, it would fetch data with `noaa_mag_api`, then reformat it into lists, then call `plot`.

The production code also has `plot_many` (several lines on one chart) and `contour_map` (maps of the field over a region). We keep just `plot` here for simplicity.

In [ ]:
import uuid
import matplotlib
matplotlib.use("Agg")  # Render to files, not to a window
import matplotlib.pyplot as plt

def save_figure_to_temp(fig) -> str:
    """Save a figure to tempimages/ and return its path."""
    os.makedirs("tempimages", exist_ok=True)
    path = os.path.join("tempimages", f"{uuid.uuid4().hex}.png")
    fig.savefig(path)
    plt.close(fig)
    return path


@tool
def plot(
    title: str,
    x: List[Union[float, int, str]],
    y: List[Union[float, int]],
    xlabel: str = "",
    ylabel: str = "",
    marker: str = "o",
) -> str:
    """
    Generalized plotting function.
    x and y should be sequences of equal length.
    Returns path to saved plot image. Once you call this tool, the plot will
    render to the user.
    """
    if len(x) != len(y):
        return "Error: x and y data must have the same length."

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(x, y, marker=marker, linestyle="-", label=ylabel or "Data")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True)
    if ylabel:
        ax.legend()

    return save_figure_to_temp(fig)

## 5. The System Prompt

Tools define what the agent can do, but the system prompt defines what it should do. This text is prepended to every conversation. Two parts are worth highlighting, because both were added in response to errors during development:

- Whenever a user asks general geomagnetism questions, always use your retriever tool: Without this, weaker models often answered from memory and skipped retrieval entirely.
- The plotting workflow section: Smaller models frequently fetched the data, *described* what the plot would look like, and stopped without ever calling `plot`. 

Often, many of these statements became more redundant for larger, smarter models.

In [ ]:
system_prompt = """
You are GeoMagi, an expert AI assistant specializing in geomagnetism and Earth sciences.
CORE EXPERTISE: Geomagnetism, Earth's magnetic field, magnetic navigation, and related geophysical phenomena.

RESPONSE GUIDELINES:
1. CONTEXT USAGE:
   - Whenever a user asks you general geomagnetism questions, always use your retriever tool to fetch relevant context to help you.
   - If the context helps, great, cite it if you can
   - If the context is not helpful, you can ignore it and answer based on your own knowledge

2. ANSWER QUALITY:
   - Be specific and precise - avoid vague generalizations
   - Match detail level to question complexity
   - Include practical applications when relevant
   - Cite authoritative sources (NOAA, USGS, etc.) when appropriate

3. PRACTICAL NOTES:
   - Always specify units and coordinate systems
   - Compass correction: Add East declination (+) from magnetic to get true, subtract West declination (-) from magnetic to get true

4. PLOTTING WORKFLOW - CRITICAL:
   - When a user asks for a plot, you MUST complete the ENTIRE workflow:
     a) First, use noaa_mag_api to fetch the required data
     b) Then, IMMEDIATELY use the plot tool to generate the visualization
     c) Do NOT stop after getting the data - the user expects to see the actual plot
   - NEVER describe what a plot would look like - ALWAYS generate the actual plot
   - The workflow is not complete until the plot tool has been called and returns a file path

5. TOOL USAGE:
   - Always use OUR tool calling methods, these use NOAA APIs
   - When fetching data for plotting, organize it properly for the plotting tools
   - Remember: plot tools expect lists of numbers, not text descriptions

PERSONALITY: Professional yet approachable, enthusiastic about the subject, educational without being condescending.

IMPORTANT: When asked to create visualizations, you must ALWAYS follow through with actually calling the plotting tools. Getting data is only the first step - creating the visual is the completion of the task.
"""

## 6. Creating the Agent

We will now pull everything together (the LLM, tools, memory, and ReAct) to form our final agent

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

llm = ChatOllama(model=MODEL_NAME)
memory = MemorySaver() # to allow for chats to have continuouty

tools = [retriever_tool, noaa_mag_api, plot]

agent = create_react_agent(
    llm,
    tools,
    checkpointer=memory,
    prompt=SystemMessage(content=system_prompt),
)
print("Agent ready.")

The agent can be represented as a graph, with nodes forming the different parts of the agentic loop. This cell is optional.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Could not render diagram ({e}). Here is the ASCII version:")
    print(agent.get_graph().draw_ascii())

## 7. Running the Agent

Before the chat interface, let's run a few questions and watch each step of the loop. The helper below streams the agent's intermediate messages (you'll see the model's reasoning, its tool calls with arguments, and the tool results) and displays any plots at the end.


In [ ]:
from IPython.display import Image as IPyImage, display

def ask(question: str, thread_id: str = "notebook"):
    """Send a question to the agent, print every step, display any plots."""
    plot_paths = []

    stream = agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": thread_id}},
        stream_mode="values",
    )

    for step in stream:
        message = step["messages"][-1]
        message.pretty_print()

        # Tool results that are .png paths are plots — collect them
        if message.type == "tool" and isinstance(message.content, str):
            if message.content.endswith(".png") and os.path.exists(message.content):
                plot_paths.append(message.content)

    for path in plot_paths:
        display(IPyImage(filename=path))

Knowledge Question: Watch for the `retrieve_geomag_docs` call, the agent should search the document corpus before answering.


In [ ]:
ask("What is magnetic declination, and why does it change over time?")

Live-Data Question: This needs a real calculation, so the agent should call `noaa_mag_api`.


In [ ]:
ask("What's the magnetic declination in Boulder, CO right now?")

Follow-Up: The agent remembers the previous turn because both calls share a `thread_id`.


In [ ]:
ask("And what about in Denver?")

Complex Request: The agent must fetch a time series and then feed it into the plotting tool.


In [ ]:
ask("Could you make me a plot of the magnetic field strength in London from 1999 until now?")

## 8. Chat With It

Run the cell, type questions at the prompt, and enter `q` to quit.

Every technique from this notebook is now working together: the agent retrieves documents when questions are conceptual, calls NOAA calculators when they need live data, chains tools to build plots, and remembers your conversation as it goes.

In [ ]:
print("ChatGeo-Magi Type 'q' to quit.\n")

while (user_input := input(">>> ")) != "q":
    ask(user_input, thread_id="chat_session")
    print()

print("Goodbye!")

## 9. Where to Go From Here

You've built the core of ChatGeo-Magi.

To explore further in this repository:

| Where | What |
|---|---|
| `agentic/cgm.py` | The production version of this pipeline, organized as a class |
| `agentic/cgmwebsite.py` | A Streamlit web interface that renders the agent's reasoning steps and plots |
| `agentic/apis/noaa_apis.py` | The full tool set, including `plot_many` and `contour_map` |
| `two_llm/` | The earlier two-LLM architecture, kept for comparison |
| `evaluation/` | The scripts and results behind the paper's evaluation |

Things to try on your own:

1. Add a new tool and watch the agent start using it
2. Swap `MODEL_NAME` for another Ollama model and compare how well it follows the plotting workflow
3. Add more documents to `agentic/data/`, delete the `chroma_db_notebook/` folder, and re-index

Have fun exploring!